In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, count
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
spark = SparkSession.builder \
    .appName("CreditCardFraudDetection") \
    .master("local[*]") \
    .getOrCreate()

spark

In [7]:
!cp /creditcard.csv /content/creditcard.csv

In [8]:
file_path = "creditcard.csv"
df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)
df.show(5)

In [9]:
df.printSchema()

In [10]:
total_count = df.count()
print("Total transaction count:", total_count)

In [11]:
df.groupBy("Class").count().show()

In [12]:
##
class_distribution = df.groupBy("Class").count().toPandas()
class_distribution["ratio"] = class_distribution["count"] / class_distribution["count"].sum()
class_distribution

In [13]:
plt.figure(figsize=(6, 4))
sns.barplot(data=class_distribution, x="Class", y="count")
plt.yscale("log")
plt.title("Class Distribution - Log Scale")
plt.xlabel("Class")
plt.ylabel("Count - Log Scale")
plt.show()

In [14]:
missing_values = df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    for c in df.columns
])
missing_values.show()

In [15]:
feature_cols = [c for c in df.columns if c != "Class"]
print("Feature columns:", feature_cols)
print("Number of features:", len(feature_cols))

In [18]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)
df_assembled = assembler.transform(df)

In [19]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)
df_assembled = assembler.transform(df)

In [20]:
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

In [21]:
model_df = df_scaled.select(
    col("features"),
    col("Class").alias("label")
)
model_df.show(5)

In [22]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print("Training count:", train_df.count())
print("Test count:", test_df.count())

In [23]:
lr_baseline = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

In [24]:
lr_baseline_model = lr_baseline.fit(train_df)

In [25]:
baseline_predictions = lr_baseline_model.transform(test_df)

In [26]:
baseline_predictions.select("label", "prediction", "probability").show(10, truncate=False)

In [27]:
def evaluate_predictions(predictions, model_name):
    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy")
    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1")
    weighted_precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    weighted_recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall")

    accuracy = accuracy_evaluator.evaluate(predictions)
    weighted_f1 = f1_evaluator.evaluate(predictions)
    weighted_precision = weighted_precision_evaluator.evaluate(predictions)
    weighted_recall = weighted_recall_evaluator.evaluate(predictions)

    conf_matrix = predictions.groupBy("label", "prediction").count().collect()
    tp = fp = tn = fn = 0
    for row in conf_matrix:
        label = row["label"]
        prediction = row["prediction"]
        count_value = row["count"]
        if label == 1 and prediction == 1:
            tp = count_value
        elif label == 0 and prediction == 1:
            fp = count_value
        elif label == 0 and prediction == 0:
            tn = count_value
        elif label == 1 and prediction == 0:
            fn = count_value

    fraud_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    fraud_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fraud_f1 = 2 * fraud_precision * fraud_recall / (fraud_precision + fraud_recall) if (fraud_precision + fraud_recall) > 0 else 0

    print("====================================")
    print(model_name)
    print("====================================")
    print("Accuracy:", accuracy)
    print("Weighted Precision:", weighted_precision)
    print("Weighted Recall:", weighted_recall)
    print("Weighted F1-score:", weighted_f1)
    print("------------------------------------")
    print("Confusion Matrix")
    print("TP:", tp, "FP:", fp, "TN:", tn, "FN:", fn)
    print("------------------------------------")
    print("Fraud Precision:", fraud_precision)
    print("Fraud Recall:", fraud_recall)
    print("Fraud F1-score:", fraud_f1)

    return {"model": model_name, "accuracy": accuracy, "weighted_precision": weighted_precision,
            "weighted_recall": weighted_recall, "weighted_f1": weighted_f1,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "fraud_precision": fraud_precision, "fraud_recall": fraud_recall, "fraud_f1": fraud_f1}

In [28]:
baseline_results = evaluate_predictions(baseline_predictions, "Baseline Logistic Regression")

In [29]:
#Model 32 fraud işlemi kaçırdı (FN=32). Recall %65 — yani fraudların %35'ini yakalayamadı. Bu yüzden Weighted model ile bunu düzeltelim:
train_total = train_df.count()
train_fraud = train_df.filter(col("label") == 1).count()
train_normal = train_df.filter(col("label") == 0).count()

num_classes = 2
weight_normal = train_total / (num_classes * train_normal)
weight_fraud = train_total / (num_classes * train_fraud)

print("Train total:", train_total)
print("Train normal:", train_normal)
print("Train fraud:", train_fraud)
print("Weight for normal class:", weight_normal)
print("Weight for fraud class:", weight_fraud)

In [30]:
train_weighted = train_df.withColumn(
    "classWeight",
    when(col("label") == 1, weight_fraud).otherwise(weight_normal)
)
train_weighted.select("label", "classWeight").show(10)

In [31]:
lr_weighted = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)

In [32]:
lr_weighted_model = lr_weighted.fit(train_weighted)

In [33]:
weighted_predictions = lr_weighted_model.transform(test_df)

In [34]:
weighted_results = evaluate_predictions(weighted_predictions, "Weighted Logistic Regression")

In [35]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)
rf_results = evaluate_predictions(rf_predictions, "Random Forest")

In [36]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    seed=42
)

gbt_model = gbt.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)
gbt_results = evaluate_predictions(gbt_predictions, "GBT")

In [37]:
results_df = pd.DataFrame([
    baseline_results,
    weighted_results,
    rf_results,
    gbt_results
])

comparison_table = results_df[[
    "model", "accuracy", "fraud_precision",
    "fraud_recall", "fraud_f1", "tp", "fp", "tn", "fn"
]]

comparison_table_rounded = comparison_table.copy()
for metric in ["accuracy", "fraud_precision", "fraud_recall", "fraud_f1"]:
    comparison_table_rounded[metric] = comparison_table_rounded[metric].round(4)

comparison_table_rounded

In [38]:
metrics_melted = comparison_table_rounded[["model","fraud_precision","fraud_recall","fraud_f1"]].melt(id_vars="model", var_name="metric", value_name="score")
plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_melted, x="metric", y="score", hue="model")
plt.title("Fraud Class Metrics Comparison - All Models")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.show()

In [39]:
#results_df = pd.DataFrame([baseline_results, weighted_results])
#comparison_table = results_df[["model","accuracy","fraud_precision","fraud_recall","fraud_f1","tp","fp","tn","fn"]]
#comparison_table_rounded = comparison_table.copy()
#for metric in ["accuracy", "fraud_precision", "fraud_recall", "fraud_f1"]:
#    comparison_table_rounded[metric] = comparison_table_rounded[metric].round(4)
#comparison_table_rounded

In [40]:
#metrics_melted = comparison_table_rounded[["model","fraud_precision","fraud_recall","fraud_f1"]].melt(id_vars="model", var_name="metric", value_name="score")
#plt.figure(figsize=(8, 5))
#sns.barplot(data=metrics_melted, x="metric", y="score", hue="model")
#plt.title("Fraud Class Metrics Comparison")
#plt.ylim(0, 1)
#plt.show()

In [41]:
from google.colab import drive
drive.mount('/content/drive')

In [42]:
import os
os.makedirs("/content/drive/MyDrive/FraudProject", exist_ok=True)
print("Klasör oluşturuldu!")

In [43]:
rf_model.save("/content/drive/MyDrive/FraudProject/fraud_rf_model")
print("Random Forest model saved!")

In [44]:
from pyspark.ml.classification import RandomForestClassificationModel

loaded_model = RandomForestClassificationModel.load("/content/drive/MyDrive/FraudProject/fraud_rf_model")
print("Random Forest model loaded!")

In [ ]:
loaded_predictions = loaded_model.transform(test_df)
loaded_predictions.select("label", "prediction", "probability").show(10, truncate=False)

In [45]:
rf_predictions.select("label", "prediction") \
    .toPandas() \
    .to_csv("/content/drive/MyDrive/FraudProject/rf_predictions.csv", index=False)
print("RF Predictions saved!")

In [46]:
spark.stop()
print("Spark session stopped.")

In [47]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git config --global user.email "eekarapinar@gmail.com"
!git config --global user.name "elifkarapinar"

In [ ]:
!git init


In [ ]:
%%writefile .gitignore
creditcard.csv
drive/
sample_data/
*.gslides
*.gsheet
*.gdoc
__pycache__/
*.pyc

In [ ]:
!git add .gitignore
!git add fraud_detection_phase2.ipynb
!git commit -m "fraud detection phase2"
!git branch -M main
!git push -u origin main

In [ ]:
!ls /content/

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/fraud_detection_phase2.ipynb" /content/
!git add fraud_detection_phase2.ipynb
!git commit -m "add notebook"
!git push origin main

In [ ]:
!find /content/drive/MyDrive -name "fraud_detection_phase2.ipynb"

In [ ]:
import json

with open("/content/fraud_detection_phase2.ipynb", "r") as f:
    nb = json.load(f)

# Tüm hücrelerdeki çıktıları temizle
for cell in nb['cells']:
    if 'outputs' in cell:
        cell['outputs'] = []
    if 'source' in cell:
        cell['source'] = [line for line in cell['source']

with open("/content/fraud_detection_phase2.ipynb", "w") as f:
    json.dump(nb, f)

print("Temizlendi!")

In [ ]:
!git log

In [ ]:
!git push origin main

In [ ]:
!git rm --cached fraud_detection_phase2.ipynb
!git commit -m "remove notebook with secrets"
!git push origin main

In [ ]:
!git checkout --orphan newbranch
!git add .gitignore
!git commit -m "initial commit"
!git branch -D main
!git branch -m main
!git push -f origin main

In [ ]:
import json

with open("/content/fraud_detection_phase2.ipynb", "r") as f:
    nb = json.load(f)

# Token içeren hücreleri temizle
for cell in nb['cells']:
    if 'source' in cell:
        cell['source'] = [line for line in cell['source']
    if 'outputs' in cell:
        cell['outputs'] = []

with open("/content/fraud_detection_phase2.ipynb", "w") as f:
    json.dump(nb, f)

print("Temizlendi!")

In [ ]:
!git add fraud_detection_phase2.ipynb
!git commit -m "add notebook"
!git push origin main

In [ ]:
import json

with open("/content/fraud_detection_phase2.ipynb", "r") as f:
    nb = json.load(f)

# spark.stop()'tan sonraki hücreleri sil
new_cells = []
for cell in nb['cells']:
    source = ''.join(cell.get('source', []))
        continue  # Bu hücreyi atla
    if 'outputs' in cell:
        cell['outputs'] = []
    new_cells.append(cell)

nb['cells'] = new_cells

with open("/content/fraud_detection_phase2.ipynb", "w") as f:
    json.dump(nb, f)

print("Temizlendi!")

In [ ]:
!git add fraud_detection_phase2.ipynb
!git commit -m "add clean notebook"
!git push origin main